In [1]:
import sys
print(sys.executable)

D:\devhub-advanced\chat-with-pdf-rag\.venv\Scripts\python.exe


In [5]:
import langchain
import langchain_community
import langchain_text_splitters

print("LangChain:", langchain.__version__)
print("Community Imported Successfully")
print("Text Splitters Imported Successfully")

LangChain: 1.3.13
Community Imported Successfully
Text Splitters Imported Successfully


In [6]:
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

print("All Imports Successful")

All Imports Successful


In [7]:
loader = PyPDFLoader("data/the-psychology-of-money-by-morgan-housel.pdf")

documents = loader.load()

print("Total Pages:", len(documents))

documents[0]

Total Pages: 253


Document(metadata={'producer': 'calibre (4.23.0) [https://calibre-ebook.com]', 'creator': 'calibre (4.23.0) [https://calibre-ebook.com]', 'creationdate': '2020-10-14T23:29:37+00:00', 'author': 'Morgan Housel', 'keywords': 'Business & Economics, Personal Finance, General, Investing', 'moddate': '2020-10-15T02:29:37+03:00', 'title': 'The Psychology of Money: Timeless Lessons on Wealth, Greed, and Happiness', 'source': 'data/the-psychology-of-money-by-morgan-housel.pdf', 'total_pages': 253, 'page': 0, 'page_label': '1'}, page_content='')

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

chunks[0]

Total Chunks: 776


Document(metadata={'producer': 'calibre (4.23.0) [https://calibre-ebook.com]', 'creator': 'calibre (4.23.0) [https://calibre-ebook.com]', 'creationdate': '2020-10-14T23:29:37+00:00', 'author': 'Morgan Housel', 'keywords': 'Business & Economics, Personal Finance, General, Investing', 'moddate': '2020-10-15T02:29:37+03:00', 'title': 'The Psychology of Money: Timeless Lessons on Wealth, Greed, and Happiness', 'source': 'data/the-psychology-of-money-by-morgan-housel.pdf', 'total_pages': 253, 'page': 3, 'page_label': '4'}, page_content='For\nMy parents, who teach me.\nGretchen, who guides me.\nMiles and Reese, who inspire me.')

In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully")

D:\Temp\ipykernel_7088\893028751.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
D:\devhub-advanced\chat-with-pdf-rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ELYSIUM\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see 

Embedding Model Loaded Successfully


In [10]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS Vector Database Created Successfully")

FAISS Vector Database Created Successfully


In [11]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS Vector Database Created Successfully")

FAISS Vector Database Created Successfully


In [12]:
query = "What is the main idea of the book?"

docs = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(docs):
    print(f"\n===== Result {i+1} =====")
    print(doc.page_content)


===== Result 1 =====
reading material in this book can be accepted by the
Publisher, by the Author, or by the employers of the Author.

===== Result 2 =====
What you’re holding is 20 chapters, each describing what I consider to be the mostimportant and often counterintuitive features of the psychology of money. The chaptersrevolve around a common theme, but exist on their own and can be read independently.
It’s not a long book. You’re welcome. Most readers don’t ﬁnish the books they beginbecause most single topics don’t require 300 pages of explanation. I’d rather make 20 shortpoints you ﬁnish than one long one you give up on.
On we go.

===== Result 3 =====
theories, making them so popular as to render their
potential useless. Jason Zweig—who annotated a later
version of Graham’s book—once wrote:
 
Graham was constantly experimenting and retesting his
assumptions and seeking out what works—not what worked
yesterday but what works today. In each revised edition of
The Intelligent Inve

In [13]:
from google import genai

client = genai.Client(
    api_key="AQ.Ab8RN6IXSGhG3ZFyTOclxHZLfFq4CjQExBYRpJbnqqcK2NPF3A"
)

print("Gemini Ready")

Gemini Ready


In [14]:
question = "What is the main lesson of this book?"

In [15]:
docs = vectorstore.similarity_search(question, k=3)

context = "\n\n".join([doc.page_content for doc in docs])

print(context)

reading material in this book can be accepted by the
Publisher, by the Author, or by the employers of the Author.

What you’re holding is 20 chapters, each describing what I consider to be the mostimportant and often counterintuitive features of the psychology of money. The chaptersrevolve around a common theme, but exist on their own and can be read independently.
It’s not a long book. You’re welcome. Most readers don’t ﬁnish the books they beginbecause most single topics don’t require 300 pages of explanation. I’d rather make 20 shortpoints you ﬁnish than one long one you give up on.
On we go.

The premise of this book is that doing well with money has a little to do with how smart youare and a lot to do with how you behave. And behavior is hard to teach, even to really smartpeople.


In [18]:
import time

for i in range(5):
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        print(response.text)
        break

    except Exception as e:
        print("Retrying...", e)
        time.sleep(5)
        

Retrying... 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
Retrying... 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
Retrying... 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
Retrying... 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
Retrying

In [19]:
from google import genai

client = genai.Client(api_key="AQ.Ab8RN6IXSGhG3ZFyTOclxHZLfFq4CjQExBYRpJbnqqcK2NPF3A")

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [20]:
from google import genai

client = genai.Client(
    api_key="AQ.Ab8RN6IXSGhG3ZFyTOclxHZLfFq4CjQExBYRpJbnqqcK2NPF3A"
)

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Hello"
)

print(response.text)

Hello! How can I help you today?


In [21]:
import google.genai as genai
import google

print(genai.__version__)

2.11.0


In [22]:
query = "What is the psychology of money?"

docs = vectorstore.similarity_search(query, k=3)

print("Retrieved Chunks:", len(docs))

for i, doc in enumerate(docs):
    print(f"\nChunk {i+1}\n")
    print(doc.page_content[:500])

Retrieved Chunks: 3

Chunk 1

I love Voltaire’s observation that “History never repeats itself; man always does.” It appliesso well to how we behave with money.
In 2018, I wrote a report outlining 20 of the most important ﬂaws, biases, and causes ofbad behavior I’ve seen aﬀect people when dealing with money. It was called The Psychologyof Money, and over one million people have read it. This book is a deeper dive into thetopic. Some short passages from the report appear unaltered in this book.

Chunk 2

I call this soft skill the psychology of money. The aim of this book is to use short stories toconvince you that soft skills are more important than the technical side of money. I’ll do thisin a way that will help everyone—from Read to Fuscone and everyone in between—makebetter ﬁnancial decisions.
These soft skills are, I’ve come to realize, greatly underappreciated.

Chunk 3

My own appreciation for the psychology of money is shaped by more than a decade ofwriting on the topic. I began

In [24]:
print(len(prompt))

1162


In [25]:
docs = vectorstore.similarity_search(query, k=2)

context = "\n\n".join([doc.page_content for doc in docs])

In [26]:
print(len(context))

839


In [28]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

print(response.text)

The psychology of money is a soft skill, which the author aims to show is more important than the technical side of money.


In [29]:
vectorstore.save_local("faiss_index")

print("FAISS Saved Successfully")

FAISS Saved Successfully


In [30]:
new_db = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS Loaded Successfully")

FAISS Loaded Successfully


In [32]:
while True:
    query = input("Ask Question: ")

    if query.lower() == "exit":
        break

    docs = new_db.similarity_search(query, k=2)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
Answer only from the given context.

Context:
{context}

Question:
{query}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    print("\nAnswer:")
    print(response.text)
    print("-" * 50)

Ask Question:  What does the author say about Warren Buffett?



Answer:
Warren Buffett expressed that he is pledged to always run Berkshire Hathaway with more than ample cash and would not trade a night’s sleep for the chance of extra profits. Additionally, the text notes that he was mentored by Benjamin Graham.
--------------------------------------------------


Ask Question:  How does the author explain the power of compounding?



Answer:
The author explains the power of compounding by comparing it to planting oak trees: a year of growth will never show much progress, 10 years can make a meaningful difference, and 50 years can create something absolutely extraordinary. Additionally, the author notes that it requires giving an asset years and years to grow and surviving the unpredictable ups and downs experienced over time.
--------------------------------------------------


Ask Question:  Which examples are used to explain financial success?


...

Ask Question:  what is psycology of money?



Answer:
The Psychology of Money is a report written in 2018 that outlines 20 of the most important flaws, biases, and causes of bad behavior that affect people when dealing with money. It is also the title of a book that serves as a deeper dive into that same topic.
--------------------------------------------------


Ask Question:  exit()



Answer:
I am sorry, but the provided context does not contain information about "exit()".
--------------------------------------------------


Ask Question:  exit


In [35]:
def ask_pdf(question):
    docs = vectorstore.similarity_search(question, k=2)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
Use ONLY the context below to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=prompt
)

    return response.text

In [36]:
import gradio as gr

demo = gr.Interface(
    fn=ask_pdf,
    inputs=gr.Textbox(label="Ask a Question"),
    outputs=gr.Textbox(label="Answer"),
    title="Chat with PDF using RAG",
    description="Ask anything about The Psychology of Money PDF."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
